In [4]:
import pandas as pd
import sqlite3
from pathlib import Path

# 1. Rutas y Carga (Igual que antes)
data_path = Path("data")
df_netflix = pd.read_csv(data_path / 'netflix_titles.csv')
df_amazon = pd.read_csv(data_path / 'amazon_prime_titles.csv')
df_disney = pd.read_csv(data_path / 'disney_plus_titles.csv')
df_apple = pd.read_csv(data_path / 'appletv_titles.csv')
df_hbo = pd.read_csv(data_path / 'HBO_MAX_Content.csv').rename(columns={'year': 'release_year'})

# 2. Etiquetado
df_netflix['platform'] = 'Netflix'
df_amazon['platform'] = 'Amazon'
df_disney['platform'] = 'Disney'
df_apple['platform'] = 'Apple TV'
df_hbo['platform'] = 'HBO Max'

# 3. EL CAMBIO: Incluir tus 132 series scrapeadas
# Nota: 'df_apple_series' es el DataFrame que creaste con el scraping hace un momento
# Lo unimos al dataframe de Apple antes de la unión total
df_apple_completo = pd.concat([df_apple, df_apple_series], ignore_index=True)

# 4. Selección de columnas y Unión Final
cols = ['title', 'type', 'release_year', 'platform']

df_total = pd.concat([
    df_netflix[cols], 
    df_amazon[cols], 
    df_disney[cols], 
    df_apple_completo[cols], # <--- Usamos el enriquecido
    df_hbo[cols]
], ignore_index=True)

# 5. Guardado Limpio en SQL
conn = sqlite3.connect("streaming_project.db")
df_total.to_sql("content", conn, if_exists="replace", index=False)

print(f"✅ ¡Proyecto Reseteado! Base de datos sólida con {len(df_total)} títulos.")
print(f"🍎 Apple TV ahora tiene {len(df_total[df_total['platform']=='Apple TV'])} títulos en total.")

✅ ¡Proyecto Reseteado! Base de datos sólida con 22232 títulos.
🍎 Apple TV ahora tiene 302 títulos en total.


In [ ]:
# Ver si hay títulos repetidos en Apple TV
query_duplicados = """
SELECT title, COUNT(*) as repeticiones
FROM content
WHERE platform = 'Apple TV'
GROUP BY title
HAVING repeticiones > 1
ORDER BY repeticiones DESC
LIMIT 10;
"""
df_dup = pd.read_sql(query_duplicados, conn)
print(df_dup)

# Justo antes de guardar en SQL, elimina filas que sean exactamente iguales


              title  repeticiones
0         WeCrashed             2
1  The Morning Show             2
2         Ted Lasso             2
3           Servant             2
4               See             2


In [7]:
# 1. ¿Cuántos títulos hay por cada fuente antes de limpiar?
print(f"Registros en Apple CSV: {len(df_apple)}")
print(f"Registros en Apple Scrapeado: {len(df_apple_series)}")

# 2. Vamos a ver si hay duplicados masivos (sin límite de 10)
dups_reales = df_total[df_total['platform'] == 'Apple TV'].duplicated(subset=['title']).sum()
print(f"Total de duplicados exactos encontrados: {dups_reales}")

# 3. ¿Hay títulos que son nulos o vacíos?
vacios = df_total[df_total['platform'] == 'Apple TV']['title'].isna().sum()
print(f"Títulos vacíos o Nulos: {vacios}")

Registros en Apple CSV: 170
Registros en Apple Scrapeado: 132
Total de duplicados exactos encontrados: 0
Títulos vacíos o Nulos: 0


In [11]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import sqlite3
import time
from pathlib import Path

# --- 1. CONFIGURACIÓN Y CARGA DE HISTÓRICO ---
data_path = Path("data")
def load_csv(name, platform):
    df = pd.read_csv(data_path / name)
    if 'year' in df.columns: df = df.rename(columns={'year': 'release_year'})
    df['platform'] = platform
    return df[['title', 'type', 'release_year', 'platform']]

# Cargamos lo que tenemos
df_list = [
    load_csv('netflix_titles.csv', 'Netflix'),
    load_csv('amazon_prime_titles.csv', 'Amazon'),
    load_csv('disney_plus_titles.csv', 'Disney'),
    load_csv('appletv_titles.csv', 'Apple TV'),
    load_csv('HBO_MAX_Content.csv', 'HBO Max')
]
df_historico = pd.concat(df_list, ignore_index=True)

# --- 2. SCRAPING MASIVO PARA ACTUALIZAR A 2026 ---
plataformas_urls = {
    "Netflix": "https://www.justwatch.com/es/proveedor/netflix/series",
    "Disney": "https://www.justwatch.com/es/proveedor/disney-plus/series",
    "Amazon": "https://www.justwatch.com/es/proveedor/amazon-prime-video/series",
    "HBO Max": "https://www.justwatch.com/es/proveedor/hbo-max/series",
    "Apple TV": "https://www.justwatch.com/es/proveedor/apple-tv-plus/series"
}

def get_current_content(platform_name, url):
    headers = {"User-Agent": "Mozilla/5.0"}
    try:
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.text, 'html.parser')
        new_data = []
        for img in soup.find_all('img', alt=True):
            title = img['alt']
            if title and title not in [platform_name, 'JustWatch']:
                new_data.append({'title': title, 'type': 'TV Show', 'release_year': 2026, 'platform': platform_name})
        return pd.DataFrame(new_data)
    except:
        return pd.DataFrame()

print("🌐 Iniciando actualización de catálogos al 2026...")
scraped_list = []
for p, url in plataformas_urls.items():
    df_p = get_current_content(p, url)
    scraped_list.append(df_p)
    print(f"✅ {p} actualizado.")
    time.sleep(1) # Evitar bloqueos

# --- 3. UNIFICACIÓN Y LIMPIEZA TOTAL ---
df_actualizado = pd.concat(scraped_list, ignore_index=True)
df_final = pd.concat([df_historico, df_actualizado], ignore_index=True)

# Normalización y eliminación de duplicados (Clave para el éxito)
df_final['title'] = df_final['title'].str.strip()
df_final['type'] = df_final['type'].replace({'Tv Show': 'TV Show', 'Series': 'TV Show'})
df_final = df_final.drop_duplicates(subset=['title', 'platform'], keep='last') # 'last' para quedarnos con la versión de 2026 si existe

# --- 4. PERSISTENCIA EN SQL ---
conn = sqlite3.connect("streaming_project.db")
df_final.to_sql("content", conn, if_exists="replace", index=False)

print(f"\n🏆 ¡PROYECTO ACTUALIZADO! Total títulos: {len(df_final)}")

🌐 Iniciando actualización de catálogos al 2026...
✅ Netflix actualizado.
✅ Disney actualizado.
✅ Amazon actualizado.
✅ HBO Max actualizado.
✅ Apple TV actualizado.

🏆 ¡PROYECTO ACTUALIZADO! Total títulos: 22653
